In [1]:
!pip install langchain-community pypdf

  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 180.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 551.0 kB/s eta 0:00:00a 0:00:01
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
Using cached mypy_extensions-1.0.0-py3-none-any.whl (4.7 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.15
    Uninstalling langchain-core-0.3.15:
      Successfully uninstalled langchain-core-0.3.15
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.2
    Uninstalling langchain-text-splitters-0.3.2:
      Successfully uninstalled langchain-text-splitters-0.3.2
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.7
    Uninstalling langchain-0.3.7:
      Successfully uninstalled langchain-0.3.7

[notice] A new release of pip is available: 24.2 -> 24.3.1

In [4]:
# load_env
import os
from dotenv import load_dotenv

load_dotenv()

os.getenv("LANGCHAIN_PROJECT")

'semantic-search'

### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata
- page_content: a string representing the content
- metadata: a dictionary of metadata
- id: a unique identifier for the document


In [8]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

### Document Loaders
Document loaders are classes that implement the `load` method, which returns a list of `Document` instances.

In [10]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "docs/Token-Budget-Aware LLM Reasoning.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

11


In [11]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Token-Budget-Aware LLM Reasoning
Tingxu Han1, Chunrong Fang*1, Shiyu Zhao2,
Shiqing Ma3, Zhenyu Chen1, Zhenting Wang†2
1Nanjing University 2Rutgers University 3UMass Amherst
Abstract
Reasoning is crit

{'source': 'docs/Token-Budget-Aware LLM Reasoning.pdf', 'page': 0}


### Document Splitter ( ?? chunking)
Document splitters are classes that implement the `split` method, which returns a list of `Document` instances.


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

65

### Vector Embeddings
Vector embeddings allows numerical representation of text data, which can be used for similarity search, clustering, and other tasks.

In [13]:
!pip install -qU langchain-openai


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [14]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [15]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 3072

[-0.019284693524241447, -0.020070312544703484, -0.008478743024170399, -0.02353888936340809, 0.03489328920841217, -0.04437999427318573, -0.013259153813123703, 0.04464681074023247, -0.05093175172805786, 0.011028295382857323]


In [16]:
## Storing vectors in in-memory vector store
!pip install -qU langchain-core


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [18]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

ids = vector_store.add_documents(documents=all_splits)
ids[:5]

['7ae2821a-755d-4f0d-80e0-ca73f38148fe',
 '113ecf9e-d36d-43c6-9a79-983909ff866e',
 '97c2c53b-d84c-4921-ab58-42ae15d0426b',
 '2c426cbb-6a54-47df-b4e8-603c1784da94',
 '3814d505-ad9a-4a48-819c-ac76feba2f6d']

In [24]:
## Querying the vector store
results = vector_store.similarity_search(
    "How many token budget estimators are described in the paper?"
)

print(results[0])

page_content='Task: Analyze the given question and estimate the 
minimum number of tokens required to generate a
complete and accurate response. Please Give the
response by strictly following this format: [[budget]],
for example, Budget: [[12]].
Figure 5: The prompt for zero-shot estimator.
Regression Estimator. For regression-based esti-
mator, we aim to train/fine-tune another LLM f(θ)
to serve as the estimator, such that f(θ) estimates
the optimal token budget given a specific LLM and
a particular question. Given D = {(p, xi, β∗
i )}N
i=1,
p is the instruction prompt, xi is a question, β∗
i
is our searched optimal budget (searched by Algo-
rithm 1 and Algorithm 2) forx and N is the dataset
size. We assign the instruction prompt p as “Esti-
mate the token budget for the following question”.
Next, we initialize f(θ) using a pre-trained LLM,
such as LLaMA 3-8B (AI@Meta, 2024). Then,
we craft the target output yi for xi using β∗
i . For
example, given aβ∗
0 as 14, the corresponding targ

In [21]:
## async similarity search
#results = await vector_store.asimilarity_search("When was Nike incorporated?")

#print(results[0])

In [23]:
## similarity search with scores
results = vector_store.similarity_search_with_score("What is the best available token budget estimator?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.6013527580591261

page_content='for example, Budget: [[12]].
Budget Estimation
1. Time out: 3 hours (1 to 4 PM).
2. Distance = speed × time = 50 mph × 3 hours = 
150 miles. 3. Time back = distance ÷ speed = 150 
miles ÷ 30 mph = 5 hours. 
Answer: 5 hours.
Please answer the above question.
Let's think step by step and use less than 26 tokens:
Token-budget-aware CoT
(c) TALE (68 output tokens).
Figure 7: An intuitive example to illustrate the workflow
of TALE.
A.1 Definition of Ideal Budget Range
Ideal Budget Range. Based on the observation of
token elasticity, a token cost bottom range exists
during searching for the optimal budget. In this
range, the token costs approach the token cost low-
est bound. Before or after the range, the token cost
will increase. We define such a bottom range as
“ideal budget range”. It’s worth noting that the bud-
get continuously degrades during the search. Only
the token cost rebounds. That’s why we refer to
this observation as token elasticity. 

In [25]:
## Use embeddings for a query
embedding = embeddings.embed_query("What is the best available token budget estimator?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='for example, Budget: [[12]].
Budget Estimation
1. Time out: 3 hours (1 to 4 PM).
2. Distance = speed × time = 50 mph × 3 hours = 
150 miles. 3. Time back = distance ÷ speed = 150 
miles ÷ 30 mph = 5 hours. 
Answer: 5 hours.
Please answer the above question.
Let's think step by step and use less than 26 tokens:
Token-budget-aware CoT
(c) TALE (68 output tokens).
Figure 7: An intuitive example to illustrate the workflow
of TALE.
A.1 Definition of Ideal Budget Range
Ideal Budget Range. Based on the observation of
token elasticity, a token cost bottom range exists
during searching for the optimal budget. In this
range, the token costs approach the token cost low-
est bound. Before or after the range, the token cost
will increase. We define such a bottom range as
“ideal budget range”. It’s worth noting that the bud-
get continuously degrades during the search. Only
the token cost rebounds. That’s why we refer to
this observation as token elasticity. To summarize,' metadata={'s

### Retrievers

In [26]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "How many token budget estimators are described in the paper?",
        "What is the best available token budget estimator?",
    ],
)

[[Document(id='cbb334d0-19d0-4dc8-afac-7e2b5c55edb5', metadata={'source': 'docs/Token-Budget-Aware LLM Reasoning.pdf', 'page': 5, 'start_index': 0}, page_content='Task: Analyze the given question and estimate the \nminimum number of tokens required to generate a\ncomplete and accurate response. Please Give the\nresponse by strictly following this format: [[budget]],\nfor example, Budget: [[12]].\nFigure 5: The prompt for zero-shot estimator.\nRegression Estimator. For regression-based esti-\nmator, we aim to train/fine-tune another LLM f(θ)\nto serve as the estimator, such that f(θ) estimates\nthe optimal token budget given a specific LLM and\na particular question. Given D = {(p, xi, β∗\ni )}N\ni=1,\np is the instruction prompt, xi is a question, β∗\ni\nis our searched optimal budget (searched by Algo-\nrithm 1 and Algorithm 2) forx and N is the dataset\nsize. We assign the instruction prompt p as “Esti-\nmate the token budget for the following question”.\nNext, we initialize f(θ) usi